In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/amex_project")
PROCESSED = PROJECT / "data" / "processed"

sample = pd.read_parquet(
    PROCESSED / "customer_history_30k.parquet"
)

sample_labels = pd.read_parquet(
    PROCESSED / "customer_labels_30k.parquet"
)

split_map = pd.read_parquet(
    PROCESSED / "customer_split_30k.parquet"
)

CAT_COLS = [
    "B_30", "B_38", "D_114", "D_116", "D_117",
    "D_120", "D_126", "D_63", "D_64", "D_66", "D_68",
]

NUM_COLS = [
    col for col in sample.columns
    if col not in ["customer_ID", "S_2"] + CAT_COLS
]

sample["S_2"] = pd.to_datetime(sample["S_2"])

sample = sample.sort_values(
    ["customer_ID", "S_2"]
).reset_index(drop=True)

assert not sample.duplicated(["customer_ID", "S_2"]).any()
assert sample["S_2"].notna().all()
assert sample_labels["customer_ID"].is_unique
assert split_map["customer_ID"].is_unique

assert (
    set(sample["customer_ID"])
    == set(sample_labels["customer_ID"])
    == set(split_map["customer_ID"])
)

print("이력:", sample.shape)

Mounted at /content/drive
이력: (361527, 190)


In [2]:
grouped = sample.groupby("customer_ID", sort=False)

sample["step"] = grouped.cumcount()

sample["days_before_last"] = (
    grouped["S_2"].transform("max") - sample["S_2"]
).dt.days

lengths = grouped.size()

MAX_LEN = 13
assert lengths.max() <= MAX_LEN

print("고객별 기록 수:")
display(lengths.describe())

example_id = sample["customer_ID"].iloc[0]

display(
    sample.loc[
        sample["customer_ID"].eq(example_id),
        ["customer_ID", "S_2", "step", "days_before_last"],
    ]
)

고객별 기록 수:


,0
count,30000.000000
mean,12.050900
std,2.622392
min,1.000000
25%,13.000000
50%,13.000000
75%,13.000000
max,13.000000


,customer_ID,S_2,step,days_before_last
0,00001b22f846c82c51f6e3958ccd81970162bae8b007e8...,2017-03-11,0,366
1,00001b22f846c82c51f6e3958ccd81970162bae8b007e8...,2017-04-11,1,335
2,00001b22f846c82c51f6e3958ccd81970162bae8b007e8...,2017-05-12,2,304
3,00001b22f846c82c51f6e3958ccd81970162bae8b007e8...,2017-06-10,3,275
4,00001b22f846c82c51f6e3958ccd81970162bae8b007e8...,2017-07-12,4,243
5,00001b22f846c82c51f6e3958ccd81970162bae8b007e8...,2017-08-12,5,212
6,00001b22f846c82c51f6e3958ccd81970162bae8b007e8...,2017-09-11,6,182
7,00001b22f846c82c51f6e3958ccd81970162bae8b007e8...,2017-10-12,7,151
8,00001b22f846c82c51f6e3958ccd81970162bae8b007e8...,2017-11-11,8,121
9,00001b22f846c82c51f6e3958ccd81970162bae8b007e8...,2017-12-12,9,90


In [3]:
customer_ids = np.sort(sample["customer_ID"].unique())
n_customers = len(customer_ids)

customer_to_row = pd.Series(
    np.arange(n_customers),
    index=customer_ids,
)

customer_row = (
    sample["customer_ID"]
    .map(customer_to_row)
    .to_numpy(dtype=np.int64)
)

step_index = sample["step"].to_numpy(dtype=np.int64)

# True: 실제 관측 기록 / False: 빈 자리
valid_mask = np.zeros(
    (n_customers, MAX_LEN),
    dtype=bool,
)

valid_mask[customer_row, step_index] = True

# 실제 날짜 간격도 같은 위치에 저장
time_days = np.zeros(
    (n_customers, MAX_LEN),
    dtype=np.float32,
)

time_days[customer_row, step_index] = (
    sample["days_before_last"].to_numpy(dtype=np.float32)
)

# 고객 순서에 맞춰 정답과 분할 정렬
y = (
    sample_labels.set_index("customer_ID")
    .loc[customer_ids, "target"]
    .to_numpy(dtype=np.float32)
)

split_values = (
    split_map.set_index("customer_ID")
    .loc[customer_ids, "split"]
    .to_numpy()
)

assert set(split_values) == {"train", "valid", "holdout"}

train_idx = np.flatnonzero(split_values == "train")
valid_idx = np.flatnonzero(split_values == "valid")
holdout_idx = np.flatnonzero(split_values == "holdout")

assert valid_mask.sum() == len(sample)
assert valid_mask.any(axis=1).all()

print("기록 마스크:", valid_mask.shape)
print("시간 간격:", time_days.shape)
print("고객 정답:", y.shape)

print(
    f"학습 {len(train_idx):,}명 / "
    f"검증 {len(valid_idx):,}명 / "
    f"최종 평가 {len(holdout_idx):,}명"
)

기록 마스크: (30000, 13)
시간 간격: (30000, 13)
고객 정답: (30000,)
학습 18,000명 / 검증 6,000명 / 최종 평가 6,000명


In [4]:
# 각 이력 행이 학습 고객에게 속하는지 표시
train_row_mask = sample["customer_ID"].isin(
    customer_ids[train_idx]
)

print(f"학습 고객의 이력 행 수: {train_row_mask.sum():,}")

학습 고객의 이력 행 수: 216,742


In [5]:
# 무한대가 있으면 결측값으로 처리
num_values = (
    sample[NUM_COLS]
    .replace([np.inf, -np.inf], np.nan)
    .astype("float32")
)

train_num = num_values.loc[train_row_mask]

# 학습 데이터에서만 계산
num_mean = train_num.mean().fillna(0.0)

num_std = train_num.std(ddof=0)
num_std = num_std.where(
    num_std.notna() & (num_std > 1e-6),
    1.0,
)

# 결측 여부는 값을 채우기 전에 기록
missing_values = num_values.isna().to_numpy(dtype=bool)

# 스케일링 → 극단값 제한 → 결측값 채우기
scaled_values = (
    ((num_values - num_mean) / num_std)
    .clip(-10, 10)
    .fillna(0.0)
    .to_numpy(dtype=np.float32)
)

assert np.isfinite(scaled_values).all()

In [6]:
X_num = np.zeros(
    (n_customers, MAX_LEN, len(NUM_COLS)),
    dtype=np.float32,
)

X_missing = np.zeros(
    (n_customers, MAX_LEN, len(NUM_COLS)),
    dtype=bool,
)

X_num[customer_row, step_index] = scaled_values
X_missing[customer_row, step_index] = missing_values

print("수치형 입력:", X_num.shape)
print("결측 표시:", X_missing.shape)

# 중간 배열 정리
del num_values, train_num, scaled_values, missing_values

수치형 입력: (30000, 13, 177)
결측 표시: (30000, 13, 177)


In [7]:
X_cat = np.zeros(
    (n_customers, MAX_LEN, len(CAT_COLS)),
    dtype=np.int64,
)

category_maps = {}
cat_vocab_sizes = []

for j, col in enumerate(CAT_COLS):
    values = sample[col].astype("string")

    # 범주 사전은 학습 고객의 기록에서만 생성
    known_values = sorted(
        values.loc[train_row_mask].dropna().unique().tolist()
    )

    mapping = {
        value: i + 3
        for i, value in enumerate(known_values)
    }

    category_maps[col] = mapping
    cat_vocab_sizes.append(len(mapping) + 3)

    codes = values.map(mapping).fillna(2).to_numpy(dtype=np.int64)
    codes[values.isna().to_numpy()] = 1

    X_cat[customer_row, step_index, j] = codes

print("범주형 입력:", X_cat.shape)

display(pd.DataFrame({
    "변수": CAT_COLS,
    "임베딩 사전 크기": cat_vocab_sizes,
}))

범주형 입력: (30000, 13, 11)


,변수,임베딩 사전 크기
0,B_30,6
1,B_38,10
2,D_114,5
3,D_116,5
4,D_117,10
5,D_120,5
6,D_126,6
7,D_63,9
8,D_64,7
9,D_66,5


In [8]:
X_time = (
    np.log1p(time_days) / np.log1p(365.0)
).astype(np.float32)[..., None]

# 빈 자리의 시간 값은 0
X_time[~valid_mask] = 0.0

assert np.isfinite(X_num).all()
assert np.isfinite(X_time).all()
assert (X_cat[~valid_mask] == 0).all()
assert (X_cat[valid_mask] >= 1).all()

for j, vocab_size in enumerate(cat_vocab_sizes):
    assert X_cat[:, :, j].max() < vocab_size

arrays = {
    "X_num": X_num,
    "X_missing": X_missing,
    "X_cat": X_cat,
    "X_time": X_time,
    "valid_mask": valid_mask,
    "y": y,
}

for name, arr in arrays.items():
    print(
        f"{name:12s} shape={str(arr.shape):22s} "
        f"memory={arr.nbytes / 1024**2:.1f} MB"
    )

print(
    "\n입력 배열 합계:",
    f"{sum(a.nbytes for a in arrays.values()) / 1024**2:.1f} MB"
)

X_num        shape=(30000, 13, 177)       memory=263.3 MB
X_missing    shape=(30000, 13, 177)       memory=65.8 MB
X_cat        shape=(30000, 13, 11)        memory=32.7 MB
X_time       shape=(30000, 13, 1)         memory=1.5 MB
valid_mask   shape=(30000, 13)            memory=0.4 MB
y            shape=(30000,)               memory=0.1 MB

입력 배열 합계: 363.9 MB


In [9]:
import joblib

TRANSFORMER_DIR = PROJECT / "transformer"
TRANSFORMER_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    {
        "num_cols": NUM_COLS,
        "cat_cols": CAT_COLS,
        "num_mean": num_mean,
        "num_std": num_std,
        "category_maps": category_maps,
        "cat_vocab_sizes": cat_vocab_sizes,
        "max_len": MAX_LEN,
        "clip_range": (-10, 10),
        "time_transform": "log1p(days_before_last) / log1p(365)",
    },
    TRANSFORMER_DIR / "preprocessing.joblib",
)

print("전처리 설정 저장 완료")

전처리 설정 저장 완료


In [10]:
import random
import time
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("학습 장치:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU에서도 실행되지만 학습이 느릴 수 있습니다.")


class AmexDataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        i = self.indices[index]

        return {
            "num": torch.from_numpy(X_num[i]),
            "missing": torch.from_numpy(X_missing[i]).float(),
            "cat": torch.from_numpy(X_cat[i]),
            "time": torch.from_numpy(X_time[i]),
            "valid": torch.from_numpy(valid_mask[i]),
            "target": torch.tensor(y[i], dtype=torch.float32),
        }


train_loader = DataLoader(
    AmexDataset(train_idx),
    batch_size=256,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    generator=torch.Generator().manual_seed(SEED),
)

valid_loader = DataLoader(
    AmexDataset(valid_idx),
    batch_size=512,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

학습 장치: cuda
GPU: Tesla T4


In [11]:
class AmexTransformer(nn.Module):
    def __init__(
        self,
        n_numeric,
        vocab_sizes,
        max_len=13,
        d_model=64,
        n_heads=4,
        n_layers=2,
        dropout=0.2,
    ):
        super().__init__()

        cat_dim = 8

        self.cat_embeddings = nn.ModuleList([
            nn.Embedding(size, cat_dim, padding_idx=0)
            for size in vocab_sizes
        ])

        # 수치값 + 결측 표시 + 범주 임베딩 + 시간
        input_dim = (
            n_numeric * 2
            + len(vocab_sizes) * cat_dim
            + 1
        )

        self.input_projection = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.position_embedding = nn.Embedding(
            max_len, d_model
        )

        # 각 층을 별도로 생성해 독립적으로 초기화
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=d_model * 4,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
            for _ in range(n_layers)
        ])

        self.output_norm = nn.LayerNorm(d_model)

        self.head = nn.Sequential(
            nn.Linear(d_model * 2, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, num, missing, cat, time, valid):
        cat_vectors = [
            embedding(cat[:, :, j])
            for j, embedding in enumerate(self.cat_embeddings)
        ]

        inputs = torch.cat(
            [num, missing, *cat_vectors, time],
            dim=-1,
        )

        h = self.input_projection(inputs)

        positions = torch.arange(
            h.size(1), device=h.device
        )

        h = h + self.position_embedding(positions)[None, :, :]

        # PyTorch에서는 True가 "무시할 자리"
        padding_mask = ~valid

        for layer in self.layers:
            h = layer(
                h,
                src_key_padding_mask=padding_mask,
            )

        h = self.output_norm(h)

        # 실제 기록만 평균
        weights = valid.unsqueeze(-1).to(h.dtype)

        mean_output = (
            (h * weights).sum(dim=1)
            / weights.sum(dim=1).clamp_min(1)
        )

        # 실제 기록은 앞에서부터 채웠으므로 마지막 위치 = 기록 수 - 1
        last_position = valid.sum(dim=1) - 1

        last_output = h[
            torch.arange(h.size(0), device=h.device),
            last_position,
        ]

        customer_vector = torch.cat(
            [last_output, mean_output],
            dim=-1,
        )

        # 확률이 아닌 logit 출력
        return self.head(customer_vector).squeeze(-1)


model_config = {
    "n_numeric": len(NUM_COLS),
    "vocab_sizes": cat_vocab_sizes,
    "max_len": MAX_LEN,
    "d_model": 64,
    "n_heads": 4,
    "n_layers": 2,
    "dropout": 0.2,
}

model = AmexTransformer(**model_config).to(device)

print(
    "학습 파라미터 수:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)

학습 파라미터 수: 138,417


In [12]:
def amex_metric(y_true, y_score):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_score = np.asarray(y_score, dtype=np.float64)

    assert np.isfinite(y_score).all()
    assert 0 < y_true.sum() < len(y_true)

    def weighted_gini(order):
        yy = y_true[order]
        ww = np.where(yy == 0, 20.0, 1.0)

        population = np.cumsum(ww) / ww.sum()
        lorentz = np.cumsum(yy * ww) / np.sum(yy * ww)

        return np.sum((lorentz - population) * ww)

    order = np.argsort(-y_score, kind="stable")

    sorted_y = y_true[order]
    weights = np.where(sorted_y == 0, 20.0, 1.0)

    selected = (
        np.cumsum(weights) <= int(0.04 * weights.sum())
    )

    capture4 = (
        sorted_y[selected].sum() / y_true.sum()
    )

    perfect_order = np.argsort(-y_true, kind="stable")

    normalized_gini = (
        weighted_gini(order)
        / weighted_gini(perfect_order)
    )

    return 0.5 * (normalized_gini + capture4)

In [13]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-3,
)

criterion = nn.BCEWithLogitsLoss()

EPOCHS = 30
PATIENCE = 6

best_score = -np.inf
bad_epochs = 0
training_log = []

BEST_PATH = TRANSFORMER_DIR / "transformer_v1_best.pt"


def move_batch(batch):
    return {
        key: value.to(device, non_blocking=True)
        for key, value in batch.items()
    }


@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    predictions = []
    targets = []

    for batch in loader:
        batch = move_batch(batch)
        target = batch.pop("target")

        logits = model(**batch)

        predictions.append(
            torch.sigmoid(logits).cpu().numpy()
        )
        targets.append(target.cpu().numpy())

    return np.concatenate(targets), np.concatenate(predictions)


for epoch in range(1, EPOCHS + 1):
    started = time.perf_counter()
    model.train()

    loss_sum = 0.0
    customer_count = 0

    for batch in train_loader:
        batch = move_batch(batch)
        target = batch.pop("target")

        optimizer.zero_grad(set_to_none=True)

        logits = model(**batch)
        loss = criterion(logits, target)

        if not torch.isfinite(loss):
            raise RuntimeError("학습 손실이 NaN/Inf입니다. 입력을 확인하세요.")

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0
        )

        optimizer.step()

        loss_sum += loss.item() * len(target)
        customer_count += len(target)

    valid_y, valid_pred = predict_loader(
        model, valid_loader
    )

    score = amex_metric(valid_y, valid_pred)
    auc = roc_auc_score(valid_y, valid_pred)

    improved = score > best_score

    if improved:
        best_score = score
        bad_epochs = 0

        torch.save(
            {
                "model_state": model.state_dict(),
                "model_config": model_config,
                "epoch": epoch,
                "valid_amex": float(score),
                "valid_auc": float(auc),
            },
            BEST_PATH,
        )
    else:
        bad_epochs += 1

    row = {
        "epoch": epoch,
        "train_loss": loss_sum / customer_count,
        "valid_amex": score,
        "valid_auc": auc,
        "seconds": time.perf_counter() - started,
    }

    training_log.append(row)

    pd.DataFrame(training_log).to_csv(
        TRANSFORMER_DIR / "transformer_v1_training_log.csv",
        index=False,
    )

    print(
        f"Epoch {epoch:02d} | "
        f"Loss {row['train_loss']:.4f} | "
        f"AMEX {score:.4f} | "
        f"AUC {auc:.4f} | "
        f"{row['seconds']:.1f}초"
        + (" ★ best" if improved else ""),
        flush=True,
    )

    if bad_epochs >= PATIENCE:
        print("Early stopping")
        break

Epoch 01 | Loss 0.3395 | AMEX 0.7333 | AUC 0.9505 | 3.4초 ★ best
Epoch 02 | Loss 0.2513 | AMEX 0.7561 | AUC 0.9540 | 5.3초 ★ best
Epoch 03 | Loss 0.2380 | AMEX 0.7627 | AUC 0.9554 | 2.4초 ★ best
Epoch 04 | Loss 0.2300 | AMEX 0.7574 | AUC 0.9556 | 1.6초
Epoch 05 | Loss 0.2250 | AMEX 0.7634 | AUC 0.9552 | 2.5초 ★ best
Epoch 06 | Loss 0.2173 | AMEX 0.7414 | AUC 0.9548 | 2.5초
Epoch 07 | Loss 0.2128 | AMEX 0.7493 | AUC 0.9543 | 5.1초
Epoch 08 | Loss 0.2087 | AMEX 0.7500 | AUC 0.9546 | 4.2초
Epoch 09 | Loss 0.2053 | AMEX 0.7414 | AUC 0.9534 | 1.8초
Epoch 10 | Loss 0.1992 | AMEX 0.7558 | AUC 0.9547 | 1.7초
Epoch 11 | Loss 0.1942 | AMEX 0.7434 | AUC 0.9522 | 1.6초
Early stopping


In [14]:
checkpoint = torch.load(
    BEST_PATH,
    map_location=device,
    weights_only=True,
)

model.load_state_dict(checkpoint["model_state"])

valid_y, valid_pred = predict_loader(model, valid_loader)

transformer_results = pd.DataFrame({
    "customer_ID": customer_ids[valid_idx],
    "target": valid_y.astype(int),
    "risk_score": valid_pred,
})

transformer_results.to_parquet(
    TRANSFORMER_DIR / "transformer_v1_valid_predictions.parquet",
    index=False,
)

print("선택된 epoch:", checkpoint["epoch"])
print(f"검증 AMEX: {amex_metric(valid_y, valid_pred):.4f}")
print(f"검증 ROC-AUC: {roc_auc_score(valid_y, valid_pred):.4f}")

선택된 epoch: 5
검증 AMEX: 0.7634
검증 ROC-AUC: 0.9552


In [15]:
baseline_results = pd.read_parquet(
    PROJECT / "reports" / "latest_only_valid_predictions.parquet"
)

assert set(baseline_results["customer_ID"]) == set(
    transformer_results["customer_ID"]
)

paired = baseline_results.merge(
    transformer_results,
    on="customer_ID",
    suffixes=("_lgb", "_tf"),
    validate="one_to_one",
).sort_values("customer_ID")

assert (
    paired["target_lgb"] == paired["target_tf"]
).all()

comparison = pd.DataFrame([
    {
        "model": name,
        "AMEX": amex_metric(paired["target_lgb"], paired[col]),
        "ROC_AUC": roc_auc_score(paired["target_lgb"], paired[col]),
    }
    for name, col in [
        ("LightGBM baseline", "risk_score_lgb"),
        ("Transformer v1", "risk_score_tf"),
    ]
])

display(comparison.round(5))

comparison.to_csv(
    TRANSFORMER_DIR / "transformer_v1_vs_baseline.csv",
    index=False,
)

,model,AMEX,ROC_AUC
0,LightGBM baseline,0.75309,0.95516
1,Transformer v1,0.76342,0.95520


# 제출

In [16]:
%pip install -q kaggle

In [18]:
import subprocess
from pathlib import Path

RAW = Path("/content/amex/raw")
RAW.mkdir(parents=True, exist_ok=True)

import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Kaggle API 토큰 입력: ")

for filename in ["sample_submission.csv", "test_data.csv"]:
    subprocess.run(
        [
            "kaggle", "competitions", "download",
            "amex-default-prediction",
            "-f", filename,
            "-p", str(RAW),
        ],
        check=True,
    )


def get_file(filename):
    for path in [RAW / filename, RAW / f"{filename}.zip"]:
        if path.exists():
            return path
    raise FileNotFoundError(filename)

Kaggle API 토큰 입력: ··········


In [19]:
import numpy as np
import pandas as pd
import torch
import joblib

PROJECT = Path("/content/drive/MyDrive/amex_project")
TRANSFORMER_DIR = PROJECT / "transformer"

prep = joblib.load(
    TRANSFORMER_DIR / "preprocessing.joblib"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint = torch.load(
    TRANSFORMER_DIR / "transformer_v1_best.pt",
    map_location="cpu",
    weights_only=True,
)

submit_model = AmexTransformer(
    **checkpoint["model_config"]
)

submit_model.load_state_dict(checkpoint["model_state"])
submit_model = submit_model.to(device).eval()

num_cols = prep["num_cols"]
cat_cols = prep["cat_cols"]
max_len = prep["max_len"]

template = pd.read_csv(
    get_file("sample_submission.csv"),
    dtype={"customer_ID": "string"},
)

assert template["customer_ID"].is_unique

print("테스트 고객 수:", f"{len(template):,}")
print("추론 장치:", device)
print("사용 모델 epoch:", checkpoint["epoch"])

테스트 고객 수: 924,621
추론 장치: cuda
사용 모델 epoch: 5


In [20]:
@torch.inference_mode()
def predict_customer_block(block):
    block = block.sort_values(
        ["customer_ID", "S_2"]
    ).reset_index(drop=True)

    assert not block.duplicated(
        ["customer_ID", "S_2"]
    ).any()
    assert block["S_2"].notna().all()

    ids = pd.Index(block["customer_ID"].unique())
    row = ids.get_indexer(block["customer_ID"])

    grouped = block.groupby("customer_ID", sort=False)
    step = grouped.cumcount().to_numpy()

    if step.max() >= max_len:
        raise ValueError("학습 시 최대 기록 수를 초과했습니다.")

    n = len(ids)

    num = np.zeros(
        (n, max_len, len(num_cols)), dtype=np.float32
    )
    missing = np.zeros_like(num)
    cat = np.zeros(
        (n, max_len, len(cat_cols)), dtype=np.int64
    )
    times = np.zeros((n, max_len, 1), dtype=np.float32)
    valid = np.zeros((n, max_len), dtype=bool)

    values = block[num_cols].replace(
        [np.inf, -np.inf], np.nan
    )

    missing[row, step] = values.isna().to_numpy(
        dtype=np.float32
    )

    scaled = (
        ((values - prep["num_mean"]) / prep["num_std"])
        .clip(*prep["clip_range"])
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )

    num[row, step] = scaled
    valid[row, step] = True

    for j, col in enumerate(cat_cols):
        values_cat = block[col].astype("string")

        codes = (
            values_cat.map(prep["category_maps"][col])
            .fillna(2)
            .to_numpy(dtype=np.int64)
        )

        codes[values_cat.isna().to_numpy()] = 1
        cat[row, step, j] = codes

    days = (
        grouped["S_2"].transform("max") - block["S_2"]
    ).dt.days.to_numpy(dtype=np.float32)

    times[row, step, 0] = (
        np.log1p(days) / np.log1p(365.0)
    )

    predictions = []

    for start in range(0, n, 512):
        end = start + 512

        batch = {
            "num": torch.from_numpy(num[start:end]).to(device),
            "missing": torch.from_numpy(missing[start:end]).to(device),
            "cat": torch.from_numpy(cat[start:end]).to(device),
            "time": torch.from_numpy(times[start:end]).to(device),
            "valid": torch.from_numpy(valid[start:end]).to(device),
        }

        logits = submit_model(**batch)

        predictions.append(
            torch.sigmoid(logits).cpu().numpy()
        )

    return pd.DataFrame({
        "customer_ID": ids,
        "prediction": np.concatenate(predictions),
    })

In [21]:
import time

dtype_map = {col: "float32" for col in num_cols}
dtype_map.update({col: "string" for col in cat_cols})
dtype_map["customer_ID"] = "string"

prediction_parts = []
carry = None
last_seen_id = None

rows_read = 0
customers_predicted = 0
started = time.perf_counter()

with pd.read_csv(
    get_file("test_data.csv"),
    dtype=dtype_map,
    parse_dates=["S_2"],
    chunksize=50_000,
) as reader:

    for chunk_no, chunk in enumerate(reader, start=1):
        ids_in_chunk = chunk["customer_ID"]

        assert ids_in_chunk.notna().all()
        assert ids_in_chunk.is_monotonic_increasing, (
            "고객 ID 정렬 확인이 필요합니다."
        )

        if last_seen_id is not None:
            assert ids_in_chunk.iloc[0] >= last_seen_id, (
                "청크 간 고객 ID 순서가 맞지 않습니다."
            )

        last_seen_id = ids_in_chunk.iloc[-1]
        rows_read += len(chunk)

        if carry is not None:
            chunk = pd.concat(
                [carry, chunk], ignore_index=True
            )

        last_customer = chunk["customer_ID"].iloc[-1]
        is_last = chunk["customer_ID"].eq(last_customer)

        carry = chunk.loc[is_last].copy()
        complete = chunk.loc[~is_last]

        if not complete.empty:
            result = predict_customer_block(complete)
            prediction_parts.append(result)
            customers_predicted += len(result)

        if chunk_no == 1 or chunk_no % 10 == 0:
            minutes = (time.perf_counter() - started) / 60

            print(
                f"읽은 행 {rows_read:,} | "
                f"예측 고객 {customers_predicted:,} | "
                f"{minutes:.1f}분",
                flush=True,
            )

# 파일의 마지막 고객 처리
if carry is not None and not carry.empty:
    prediction_parts.append(
        predict_customer_block(carry)
    )

test_predictions = pd.concat(
    prediction_parts, ignore_index=True
)

assert test_predictions["customer_ID"].is_unique

print("완료! 예측 고객 수:", f"{len(test_predictions):,}")

읽은 행 50,000 | 예측 고객 4,086 | 0.1분
읽은 행 500,000 | 예측 고객 40,694 | 0.7분
읽은 행 1,000,000 | 예측 고객 81,357 | 1.4분
읽은 행 1,500,000 | 예측 고객 122,011 | 2.0분
읽은 행 2,000,000 | 예측 고객 162,616 | 2.7분
읽은 행 2,500,000 | 예측 고객 203,364 | 3.4분
읽은 행 3,000,000 | 예측 고객 244,056 | 4.1분
읽은 행 3,500,000 | 예측 고객 284,777 | 4.8분
읽은 행 4,000,000 | 예측 고객 325,446 | 5.5분
읽은 행 4,500,000 | 예측 고객 366,175 | 6.1분
읽은 행 5,000,000 | 예측 고객 406,810 | 6.8분
읽은 행 5,500,000 | 예측 고객 447,542 | 7.5분
읽은 행 6,000,000 | 예측 고객 488,255 | 8.2분
읽은 행 6,500,000 | 예측 고객 528,889 | 8.8분
읽은 행 7,000,000 | 예측 고객 569,549 | 9.5분
읽은 행 7,500,000 | 예측 고객 610,199 | 10.1분
읽은 행 8,000,000 | 예측 고객 650,897 | 10.8분
읽은 행 8,500,000 | 예측 고객 691,579 | 11.4분
읽은 행 9,000,000 | 예측 고객 732,209 | 12.1분
읽은 행 9,500,000 | 예측 고객 772,853 | 12.7분
읽은 행 10,000,000 | 예측 고객 813,534 | 13.4분
읽은 행 10,500,000 | 예측 고객 854,295 | 14.0분
읽은 행 11,000,000 | 예측 고객 895,030 | 14.6분
완료! 예측 고객 수: 924,621


In [22]:
assert set(test_predictions["customer_ID"]) == set(
    template["customer_ID"]
), "예측 고객이 누락됐거나 추가됐습니다."

joined = template[["customer_ID"]].merge(
    test_predictions,
    on="customer_ID",
    how="left",
    sort=False,
    validate="one_to_one",
)

submission = joined.rename(
    columns={"prediction": "prediction"}
)

assert list(submission.columns) == list(template.columns)
assert len(submission) == len(template)
assert submission["customer_ID"].equals(template["customer_ID"])
assert np.isfinite(submission["prediction"]).all()
assert submission["prediction"].between(0, 1).all()

SUBMISSION_DIR = PROJECT / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

SUBMISSION_PATH = SUBMISSION_DIR / "transformer_v1_submission.csv"

submission.to_csv(
    SUBMISSION_PATH,
    index=False,
    float_format="%.9f",
)

print("저장 위치:", SUBMISSION_PATH)
display(submission.head())

저장 위치: /content/drive/MyDrive/amex_project/submissions/transformer_v1_submission.csv


,customer_ID,prediction
0,00000469ba478561f23a92a868bd366de6f6527a684c9a...,0.015599
1,00001bf2e77ff879fab36aa4fac689b9ba411dae63ae39...,0.001480
2,0000210045da4f81e5f122c6bde5c2a617d03eef67f82c...,0.006774
3,00003b41e58ede33b8daf61ab56d9952f17c9ad1c3976c...,0.264407
4,00004b22eaeeeb0ec976890c1d9bfc14fd9427e98c4ee9...,0.884433


In [23]:
subprocess.run(
    [
        "kaggle", "competitions", "submit",
        "amex-default-prediction",
        "-f", str(SUBMISSION_PATH),
        "-m", "Transformer v1 - trained on 18k customers",
    ],
    check=True,
)

CompletedProcess(args=['kaggle', 'competitions', 'submit', 'amex-default-prediction', '-f', '/content/drive/MyDrive/amex_project/submissions/transformer_v1_submission.csv', '-m', 'Transformer v1 - trained on 18k customers'], returncode=0)

In [24]:
!kaggle competitions submissions amex-default-prediction

fileName                       date                        description                                status                    publicScore  privateScore  
-----------------------------  --------------------------  -----------------------------------------  ------------------------  -----------  ------------  
transformer_v1_submission.csv  2026-09-16 13:02:55.070000  Transformer v1 - trained on 18k customers  SubmissionStatus.PENDING                             


In [25]:
from pathlib import Path
import shutil

LOCAL_RAW = Path("/content/amex/raw")
DRIVE_RAW = Path(
    "/content/drive/MyDrive/amex_project/data/raw"
)
DRIVE_RAW.mkdir(parents=True, exist_ok=True)

for filename in ["sample_submission.csv", "test_data.csv"]:
    candidates = [
        LOCAL_RAW / f"{filename}.zip",
        LOCAL_RAW / filename,
    ]

    source = next(
        (path for path in candidates if path.exists()),
        None,
    )

    if source is None:
        raise FileNotFoundError(f"다운로드 파일이 없습니다: {filename}")

    destination = DRIVE_RAW / source.name

    if (
        destination.exists()
        and destination.stat().st_size == source.stat().st_size
    ):
        print(f"이미 저장됨: {destination.name}")
        continue

    # 복사가 끝나기 전에는 임시 파일명 사용
    temporary = destination.with_name(destination.name + ".part")

    print(f"복사 중: {source.name}", flush=True)
    shutil.copyfile(source, temporary)

    assert temporary.stat().st_size == source.stat().st_size
    temporary.replace(destination)

    print(
        f"저장 완료: {destination.name} "
        f"({destination.stat().st_size / 1024**3:.2f} GB)"
    )

복사 중: sample_submission.csv.zip
저장 완료: sample_submission.csv.zip (0.03 GB)
복사 중: test_data.csv.zip
저장 완료: test_data.csv.zip (13.76 GB)
